## Import Library

In [2]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model

# Kunci seluruh random seed agar hasil stabil
np.random.seed(42)
tf.random.set_seed(42)

I0000 00:00:1790327092.618605   38202 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790327093.339733   38202 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790327097.223355   38202 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Load Dataset & Verifikasi Distribusi Awal

In [3]:
csv_file = "dataset_susu_labeled.csv"
if not os.path.exists(csv_file):
    raise FileNotFoundError(f"File {csv_file} tidak ditemukan!")

df = pd.read_csv(csv_file)
fitur_cols = ["Suhu (°C)", "R Liquid (Ohm)", "EC Raw (mS/cm)", "EC25 (mS/cm)"]

X_raw = df[fitur_cols].values.astype(np.float32)
y_grade_raw = df["label_grade"].values

# Batasi nilai maksimum shelf-life pada 360 menit (6 jam operasional harian)
df["label_shelf_life_min"] = df["label_shelf_life_min"].clip(upper=360)
y_shelf_raw = df["label_shelf_life_min"].values.astype(np.float32).reshape(-1, 1)

print(f"Total baris data: {len(df)}")
print("Distribusi kelas awal:")
print(df["label_grade"].value_counts())

Total baris data: 336
Distribusi kelas awal:
label_grade
GRADE_C    158
GRADE_A    152
GRADE_B     26
Name: count, dtype: int64


## Stratified Split 80:20 (Sebelum Augmentasi)
Membagi data uji (20%) dan data latih (80%) di awal dengan mempertahankan rasio kelas agar terhindar dari data leakage.

In [4]:
X_train_raw, X_test_raw, y_g_train_raw, y_g_test_raw, y_s_train_raw, y_s_test_raw = train_test_split(
    X_raw,
    y_grade_raw,
    y_shelf_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_grade_raw
)

print(f"Jumlah data latih: {len(X_train_raw)}")
print(f"Jumlah data uji  : {len(X_test_raw)}")
print("\nDistribusi kelas data uji (Test Set):")
print(pd.Series(y_g_test_raw).value_counts())

Jumlah data latih: 268
Jumlah data uji  : 68

Distribusi kelas data uji (Test Set):
GRADE_C    32
GRADE_A    31
GRADE_B     5
Name: count, dtype: int64


## Oversampling Moderat (Khusus Data Latih)
Menyeimbangkan kelas minoritas (GRADE_B) hanya pada data latih menggunakan jitter noise ringan tanpa merusak data uji.

In [5]:
# =====================================================================
# 4: k-NN LOCAL INTERPOLATION (TARGET 50 SAMPEL GRADE B)
# =====================================================================
df_train = pd.DataFrame(X_train_raw, columns=fitur_cols)
df_train["label_grade"] = y_g_train_raw
df_train["label_shelf_life_min"] = y_s_train_raw

df_A = df_train[df_train["label_grade"] == "GRADE_A"]
df_B = df_train[df_train["label_grade"] == "GRADE_B"]
df_C = df_train[df_train["label_grade"] == "GRADE_C"]

# Target 50 sampel: cukup untuk mencegah suppression tanpa menelan Grade C
target_B = 50
n_needed = target_B - len(df_B)

if n_needed > 0:
    synth_rows = []
    B_vals = df_B[fitur_cols].values
    B_shelf = df_B["label_shelf_life_min"].values
    n_B = len(df_B)

    # Standarisasi internal untuk penentuan jarak Euclidean yang adil
    B_norm = (B_vals - B_vals.mean(axis=0)) / (B_vals.std(axis=0) + 1e-6)

    for _ in range(n_needed):
        i = np.random.randint(0, n_B)

        # Cari 3 tetangga terdekat dalam cluster Grade B murni
        dists = np.linalg.norm(B_norm - B_norm[i], axis=1)
        dists[i] = np.inf
        k_indices = np.argsort(dists)[:min(3, n_B - 1)]
        j = np.random.choice(k_indices)

        # Interpolasi lokal terfokus (0.30 - 0.70)
        lam = np.random.uniform(0.30, 0.70)
        synth_x = B_vals[i] + lam * (B_vals[j] - B_vals[i])
        synth_s = B_shelf[i] + lam * (B_shelf[j] - B_shelf[i])
        synth_rows.append(list(synth_x) + [synth_s])

    df_B_synth = pd.DataFrame(synth_rows, columns=fitur_cols + ["label_shelf_life_min"])
    df_B_synth["label_grade"] = "GRADE_B"
    df_B_balanced = pd.concat([df_B, df_B_synth], ignore_index=True)
else:
    df_B_balanced = df_B.copy()

df_train_balanced = pd.concat([df_A, df_B_balanced, df_C], ignore_index=True).sample(frac=1, random_state=42)

X_train_final = df_train_balanced[fitur_cols].values.astype(np.float32)
y_g_train_final = df_train_balanced["label_grade"].values
y_s_train_final = df_train_balanced["label_shelf_life_min"].values.astype(np.float32).reshape(-1, 1)

print("Distribusi Data Latih Pasca-kNN Proporsional:")
print(pd.Series(y_g_train_final).value_counts())

Distribusi Data Latih Pasca-kNN Proporsional:
GRADE_C    126
GRADE_A    121
GRADE_B     50
Name: count, dtype: int64


## Normalisasi Fitur & Target Regresi
Menyesuaikan skala data input dan output ke rentang $[0, 1]$ serta mencetak parameter konstanta untuk implementasi di firmware ESP32-S3.

In [6]:
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train_final)
X_test_scaled = scaler_X.transform(X_test_raw)

scaler_y = MinMaxScaler()
y_s_train_scaled = scaler_y.fit_transform(y_s_train_final)
y_s_test_scaled = scaler_y.transform(y_s_test_raw)

print("=== KONSTANTA NORMALISASI INPUT (C++ ESP32-S3) ===")
for i, col in enumerate(fitur_cols):
    print(f"const float MIN_{i}   = {scaler_X.data_min_[i]:.6f}f; // {col}")
    print(f"const float SCALE_{i} = {scaler_X.scale_[i]:.6f}f;")
print(f"const float SHELF_MIN   = {scaler_y.data_min_[0]:.6f}f;")
print(f"const float SHELF_SCALE = {scaler_y.scale_[0]:.6f}f;")
print("==================================================")

=== KONSTANTA NORMALISASI INPUT (C++ ESP32-S3) ===
const float MIN_0   = 3.560000f; // Suhu (°C)
const float SCALE_0 = 0.031867f;
const float MIN_1   = 253.300003f; // R Liquid (Ohm)
const float SCALE_1 = 0.003509f;
const float MIN_2   = 1.598000f; // EC Raw (mS/cm)
const float SCALE_2 = 0.226655f;
const float MIN_3   = 2.565000f; // EC25 (mS/cm)
const float SCALE_3 = 0.359583f;
const float SHELF_MIN   = 0.000000f;
const float SHELF_SCALE = 0.002778f;


## One-Hot Encoding Label Target
Mengonversi label kategori mutu menjadi representasi matriks biner untuk output softmax.

In [7]:
class_order = ["GRADE_A", "GRADE_B", "GRADE_C"]

y_g_train_df = pd.get_dummies(y_g_train_final)
y_g_test_df = pd.get_dummies(y_g_test_raw)

for col in class_order:
    if col not in y_g_train_df:
        y_g_train_df[col] = 0
    if col not in y_g_test_df:
        y_g_test_df[col] = 0

y_g_train_onehot = y_g_train_df[class_order].values.astype(np.float32)
y_g_test_onehot = y_g_test_df[class_order].values.astype(np.float32)

## Definisi Arsitektur Multi-Task MLP
Membangun jaringan saraf dengan shared backbone dan dua kepala luaran terpisah (klasifikasi mutu dan regresi masa simpan).

In [8]:
inputs = layers.Input(shape=(4,), name="sensor_features")

# Shared Backbone: 32 -> 16 Units
x = layers.Dense(32, activation="relu", name="shared_dense_1")(inputs)
x = layers.Dense(16, activation="relu", name="shared_dense_2")(x)

# Head 1: Klasifikasi Grade
out_grade = layers.Dense(3, activation="softmax", name="grade_output")(x)

# Head 2: Regresi Sisa Waktu Simpan
out_shelf = layers.Dense(1, activation="linear", name="shelf_life_output")(x)

model = Model(inputs=inputs, outputs=[out_grade, out_shelf], name="MilkQualityModel")
model.summary()

E0000 00:00:1790327097.911427   38202 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "MilkQualityModel"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor_features     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_1      │ (None, 32)        │        160 │ sensor_features[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense_2      │ (None, 16)        │        528 │ shared_dense_1[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ grade_output        │ (None, 3)         │         51 │ shared_dense_2[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shelf_life_output   │ (None, 1)         │         17 │ shared_dense_2[0… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 756 (2.95 KB)

 Trainable params: 756 (2.95 KB)

 Non-trainable params: 0 (0.00 B)

## Kompilasi & Pelatihan Model
Melatih model multi-task dengan pembobotan loss seimbang serta penghentian dini (early stopping) berbasis mode='min'.

In [9]:
# =====================================================================
# 8: KOMPILASI DENGAN KESEIMBANGAN GRADIEN MULTI-TASK (DEEP LEARNING STANDARD)
# =====================================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0025),
    loss={
        "grade_output": "categorical_crossentropy",
        "shelf_life_output": "mean_absolute_error"  # Gradien L1 stabil pada target [0, 1]
    },
    loss_weights={
        "grade_output": 1.0,       # Magnitudo ~0.30 - 0.40
        "shelf_life_output": 12.0  # (12.0 * 0.028) ~0.33 -> Seimbang 1:1 dengan klasifikasi
    },
    metrics={
        "grade_output": ["accuracy"],
        "shelf_life_output": ["mae"]
    }
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=25,
        restore_best_weights=True,
        mode="min"
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=8,
        min_lr=1e-5,
        mode="min"
    )
]

history = model.fit(
    X_train_scaled,
    {
        "grade_output": y_g_train_onehot,
        "shelf_life_output": y_s_train_scaled
    },
    validation_data=(
        X_test_scaled,
        {"grade_output": y_g_test_onehot, "shelf_life_output": y_s_test_scaled}
    ),
    epochs=140,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/140
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - grade_output_accuracy: 0.3131 - grade_output_loss: 1.1506 - loss: 5.0487 - shelf_life_output_loss: 0.3201 - shelf_life_output_mae: 0.3248 - val_grade_output_accuracy: 0.4559 - val_grade_output_loss: 1.0776 - val_loss: 3.1303 - val_shelf_life_output_loss: 0.1726 - val_shelf_life_output_mae: 0.1712 - learning_rate: 0.0025
Epoch 2/140
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - grade_output_accuracy: 0.4074 - grade_output_loss: 1.0317 - loss: 2.1364 - shelf_life_output_loss: 0.0905 - shelf_life_output_mae: 0.0921 - val_grade_output_accuracy: 0.4559 - val_grade_output_loss: 0.9412 - val_loss: 1.5574 - val_shelf_life_output_loss: 0.0504 - val_shelf_life_output_mae: 0.0512 - learning_rate: 0.0025
Epoch 3/140
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - grade_output_accuracy: 0.5892 - grade_output_loss: 0.9525 - loss: 1.7367 - shelf_life_output_loss: 0.0644 - shelf_life_output_mae: 0.0654 - val_grade_output_accuracy: 0.9265 - val_grade_output_l

## Konversi ke TensorFlow Lite (.tflite)
Mengoptimalkan bobot model ke format biner TFLite menggunakan representasi dataset kalibrasi.

In [10]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
    for i in range(len(X_train_scaled)):
        yield [X_train_scaled[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

tflite_model = converter.convert()

with open("milk_quality_model.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Model TFLite berhasil dibuat: {len(tflite_model) / 1024:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmppzv3pdh0/assets


INFO:tensorflow:Assets written to: /tmp/tmppzv3pdh0/assets


Saved artifact at '/tmp/tmppzv3pdh0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='sensor_features')
Output Type:
  List[TensorSpec(shape=(None, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)]
Captures:
  140327711959952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140327711957664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140327711963472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140327711961712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140327634849104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140327634845936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140327711961184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140327634850336: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model TFLite berhasil dibuat: 5.73 KB


/home/ridho/ngoding/Maestro Fest/pelatihan/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1790327138.805500   38202 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790327138.805521   38202 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1790327138.806562   38202 reader.cc:83] Reading SavedModel from: /tmp/tmppzv3pdh0
I0000 00:00:1790327138.807691   38202 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1790327138.807718   38202 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmppzv3pdh0
I0000 00:00:1790327138.815217   38202 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1790327138.816290   38202 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1790327138.861734   38202 loader.cc:220] Running initi

## Ekspor Header C-Array (model_data.h)
Mengonversi model biner menjadi array heksadesimal C++ yang siap ditempatkan di proyek Arduino IDE / ESP-IDF.

In [11]:
hex_array = ", ".join([f"0x{b:02x}" for b in tflite_model])
header_content = f"""// Generated by tiny-susu.ipynb
#ifndef MILK_QUALITY_MODEL_DATA_H
#define MILK_QUALITY_MODEL_DATA_H

alignas(16) const unsigned char g_milk_quality_model_data[] = {{
  {hex_array}
}};
const int g_milk_quality_model_data_len = {len(tflite_model)};

#endif // MILK_QUALITY_MODEL_DATA_H
"""

with open("model_data.h", "w") as f:
    f.write(header_content)

print("Berkas 'model_data.h' berhasil diperbarui dan siap digunakan di ESP32-S3.")

Berkas 'model_data.h' berhasil diperbarui dan siap digunakan di ESP32-S3.


In [12]:
import numpy as np

# 1. Ekstraksi bobot dan bias dari model Keras
w1, b1 = model.get_layer("shared_dense_1").get_weights()
w2, b2 = model.get_layer("shared_dense_2").get_weights()
w_grade, b_grade = model.get_layer("grade_output").get_weights()
w_shelf, b_shelf = model.get_layer("shelf_life_output").get_weights()

h1_dim = w1.shape[1]
h2_dim = w2.shape[1]

def format_2d(arr):
    lines = []
    for row in arr:
        lines.append("  { " + ", ".join([f"{v:.7f}f" for v in row]) + " }")
    return ",\n".join(lines)

def format_1d(arr):
    return "  " + ", ".join([f"{v:.7f}f" for v in arr.flatten()])

# 2. Susun berkas C++ header dengan prefix penamaan unik
header_code = f"""#ifndef MILK_MODEL_WEIGHTS_H
#define MILK_MODEL_WEIGHTS_H

#include <Arduino.h>
#include <math.h>

// Arsitektur Jaringan: 4 -> {h1_dim} -> {h2_dim} -> [3 (Grade), 1 (Shelf-life)]
const int H1_DIM = {h1_dim};
const int H2_DIM = {h2_dim};

// Layer 1 (Dense {h1_dim})
const float MODEL_W1[4][{h1_dim}] = {{
{format_2d(w1)}
}};
const float MODEL_B1[{h1_dim}] = {{
{format_1d(b1)}
}};

// Layer 2 (Dense {h2_dim})
const float MODEL_W2[{h1_dim}][{h2_dim}] = {{
{format_2d(w2)}
}};
const float MODEL_B2[{h2_dim}] = {{
{format_1d(b2)}
}};

// Output Grade (Dense 3)
const float MODEL_W_GRADE[{h2_dim}][3] = {{
{format_2d(w_grade)}
}};
const float MODEL_B_GRADE[3] = {{
{format_1d(b_grade)}
}};

// Output Shelf Life (Dense 1)
const float MODEL_W_SHELF[{h2_dim}][1] = {{
{format_2d(w_shelf)}
}};
const float MODEL_B_SHELF[1] = {{
{format_1d(b_shelf)}
}};

// Fungsi Inferensi On-Device C++ Murni
inline void predictMilkModel(float in_suhu, float in_rohm, float in_ecraw, float in_ec25,
                             String &out_grade, int &out_shelf_life_min, float max_minutes = 360.0f) {{
  float x[4] = {{ in_suhu, in_rohm, in_ecraw, in_ec25 }};

  // 1. Forward Pass Layer 1 (ReLU)
  float h1[{h1_dim}];
  for (int j = 0; j < {h1_dim}; j++) {{
    float sum = MODEL_B1[j];
    for (int i = 0; i < 4; i++) sum += x[i] * MODEL_W1[i][j];
    h1[j] = (sum > 0.0f) ? sum : 0.0f;
  }}

  // 2. Forward Pass Layer 2 (ReLU)
  float h2[{h2_dim}];
  for (int j = 0; j < {h2_dim}; j++) {{
    float sum = MODEL_B2[j];
    for (int i = 0; i < {h1_dim}; i++) sum += h1[i] * MODEL_W2[i][j];
    h2[j] = (sum > 0.0f) ? sum : 0.0f;
  }}

  // 3. Head Klasifikasi Grade (Argmax)
  float best_val = -1e9f;
  int best_idx = 0;
  for (int j = 0; j < 3; j++) {{
    float sum = MODEL_B_GRADE[j];
    for (int i = 0; i < {h2_dim}; i++) sum += h2[i] * MODEL_W_GRADE[i][j];
    if (sum > best_val) {{
      best_val = sum;
      best_idx = j;
    }}
  }}
  if (best_idx == 0) out_grade = "GRADE_A";
  else if (best_idx == 1) out_grade = "GRADE_B";
  else out_grade = "GRADE_C";

  // 4. Head Regresi Sisa Waktu Simpan (Linear)
  float shelf_sum = MODEL_B_SHELF[0];
  for (int i = 0; i < {h2_dim}; i++) shelf_sum += h2[i] * MODEL_W_SHELF[i][0];
  if (shelf_sum < 0.0f) shelf_sum = 0.0f;
  
  int minutes = (int)round(shelf_sum * max_minutes);
  out_shelf_life_min = constrain(minutes, 0, (int)max_minutes);
}}

#endif // MILK_MODEL_WEIGHTS_H
"""

with open("milk_model_weights.h", "w") as f:
    f.write(header_code)

print("[OK] Berkas 'milk_model_weights.h' berhasil diperbarui dengan prefix aman!")

[OK] Berkas 'milk_model_weights.h' berhasil diperbarui dengan prefix aman!


## Evaluasi Inferensi Model TFLite pada Data Uji
Menjalankan inferensi simulasi TFLite terhadap data uji murni untuk mengukur akurasi, matriks kebingungan, error regresi, dan latensi komputasi.

In [13]:
interpreter = tf.lite.Interpreter(model_path="milk_quality_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

idx_grade_out = 0 if output_details[0]["shape"][-1] == 3 else 1
idx_shelf_out = 1 if idx_grade_out == 0 else 0

preds_grade = []
preds_shelf = []
latencies = []

label_to_idx = {name: i for i, name in enumerate(class_order)}
y_grade_test_idx = np.array([label_to_idx[g] for g in y_g_test_raw])

for sample in X_test_scaled:
    input_data = np.expand_dims(sample, axis=0).astype(input_details[0]["dtype"])
    interpreter.set_tensor(input_details[0]["index"], input_data)

    t_start = time.time()
    interpreter.invoke()
    latencies.append((time.time() - t_start) * 1000)

    out_grade = interpreter.get_tensor(output_details[idx_grade_out]["index"])
    out_shelf = interpreter.get_tensor(output_details[idx_shelf_out]["index"])

    preds_grade.append(np.argmax(out_grade))
    # Saat menginversikan hasil regresi:
    shelf_min = scaler_y.inverse_transform(out_shelf)[0][0]
    preds_shelf.append(max(0.0, shelf_min))  # Mencegah nilai minus secara logis

preds_grade = np.array(preds_grade)
preds_shelf = np.array(preds_shelf)

print("=== 1. EVALUASI KLASIFIKASI MUTU SUSU ===")
acc = np.mean(preds_grade == y_grade_test_idx) * 100
print(f"Overall Accuracy: {acc:.2f}%\n")

print("Classification Report:")
print(classification_report(y_grade_test_idx, preds_grade, target_names=class_order, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_grade_test_idx, preds_grade))

print("\n=== 2. EVALUASI REGRESI SISA WAKTU (MENIT) ===")
mae = mean_absolute_error(y_s_test_raw, preds_shelf)
rmse = np.sqrt(mean_squared_error(y_s_test_raw, preds_shelf))
print(f"Mean Absolute Error (MAE) : {mae:.2f} Menit")
print(f"Root Mean Squared Error   : {rmse:.2f} Menit")

print("\n=== 3. LATENSI INFERENSI TFLITE ===")
print(f"Rata-rata Waktu Eksekusi: {np.mean(latencies):.4f} ms per sampel")

=== 1. EVALUASI KLASIFIKASI MUTU SUSU ===
Overall Accuracy: 95.59%

Classification Report:
              precision    recall  f1-score   support

     GRADE_A       1.00      1.00      1.00        31
     GRADE_B       1.00      0.40      0.57         5
     GRADE_C       0.91      1.00      0.96        32

    accuracy                           0.96        68
   macro avg       0.97      0.80      0.84        68
weighted avg       0.96      0.96      0.95        68

Confusion Matrix:
[[31  0  0]
 [ 0  2  3]
 [ 0  0 32]]

=== 2. EVALUASI REGRESI SISA WAKTU (MENIT) ===
Mean Absolute Error (MAE) : 3.55 Menit
Root Mean Squared Error   : 8.24 Menit

=== 3. LATENSI INFERENSI TFLITE ===
Rata-rata Waktu Eksekusi: 0.0315 ms per sampel


/home/ridho/ngoding/Maestro Fest/pelatihan/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [14]:
np.savez(
    "model_weights.npz",
    w1=model.get_layer("shared_dense_1").get_weights()[0],
    b1=model.get_layer("shared_dense_1").get_weights()[1],
    w2=model.get_layer("shared_dense_2").get_weights()[0],
    b2=model.get_layer("shared_dense_2").get_weights()[1],
    w_grade=model.get_layer("grade_output").get_weights()[0],
    b_grade=model.get_layer("grade_output").get_weights()[1],
    w_shelf=model.get_layer("shelf_life_output").get_weights()[0],
    b_shelf=model.get_layer("shelf_life_output").get_weights()[1]
)
print("File 'model_weights.npz' berhasil disimpan!")

File 'model_weights.npz' berhasil disimpan!


In [15]:
# 1. Ambil 5 sampel uji acak dari data mentah
sample_indices = [0, 10, 25, 40, 60]
samples_raw = X_test_raw[sample_indices]      # Fitur mentah: [suhu, r_liquid, ec_raw, ec_25]
samples_scaled = X_test_scaled[sample_indices]

# 2. Prediksi menggunakan Keras Model
keras_pred_grade, keras_pred_shelf = model.predict(samples_scaled)

# 3. Prediksi menggunakan Bobot NumPy (.npz)
weights = np.load("model_weights.npz")
MIN_VALS = np.array([3.560000, 253.300003, 1.598000, 2.565000], dtype=np.float32)
SCALE_VALS = np.array([0.031867, 0.003509, 0.226655, 0.359583], dtype=np.float32)

print(f"{'Idx':<4} | {'Keras Grade':<12} | {'NumPy Grade':<12} | {'Keras Shelf':<12} | {'NumPy Shelf':<12} | {'Selisih Absolut':<15}")
print("-" * 75)

for i in range(len(samples_raw)):
    # Normalisasi manual NumPy
    x_norm = (samples_raw[i] - MIN_VALS) * SCALE_VALS
    
    # Forward pass NumPy
    h1 = np.maximum(0.0, np.dot(x_norm, weights['w1']) + weights['b1'])
    h2 = np.maximum(0.0, np.dot(h1, weights['w2']) + weights['b2'])
    
    np_grade_logits = np.dot(h2, weights['w_grade']) + weights['b_grade']
    np_shelf_out = np.dot(h2, weights['w_shelf']) + weights['b_shelf']
    
    grade_classes = ["GRADE_A", "GRADE_B", "GRADE_C"]
    keras_g = grade_classes[np.argmax(keras_pred_grade[i])]
    numpy_g = grade_classes[np.argmax(np_grade_logits)]
    
    keras_s = int(round(np.clip(keras_pred_shelf[i][0] * 360.0, 0, 360)))
    numpy_s = int(round(np.clip(np_shelf_out[0] * 360.0, 0, 360)))
    
    diff = abs(keras_pred_shelf[i][0] - np_shelf_out[0])
    
    print(f"{i:<4} | {keras_g:<12} | {numpy_g:<12} | {keras_s:<10} m | {numpy_s:<10} m | {diff:.8f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Idx  | Keras Grade  | NumPy Grade  | Keras Shelf  | NumPy Shelf  | Selisih Absolut
---------------------------------------------------------------------------
0    | GRADE_C      | GRADE_C      | 0          m | 0          m | 0.00000000
1    | GRADE_B      | GRADE_B      | 81         m | 81         m | 0.00004362
2    | GRADE_C      | GRADE_C      | 5          m | 5          m | 0.00000236
3    | GRADE_A      | GRADE_A      | 360        m | 360        m | 0.00002360
4    | GRADE_C      | GRADE_C      | 0          m | 0          m | 0.00000000
